# Experiment Log — D\&R Knowledge Distillation

**Project:** Debate & Reflect (D&R) for Knowledge Distillation  
**Task:** MMLU-Pro Computer Science  
**Student model:** Qwen2.5-1.5B-Instruct  
**Evaluation metric:** Accuracy on 137-question test set  

This log records all runs, hyperparameters, seeds, and results.

---

## System Configuration

| Component | Details |
|---|---|
| OS | Windows 11 + WSL2 (Ubuntu) |
| CPU | *(Intel Core i5-7500)* |
| RAM | 16 GB |
| GPU | *(not used — CPU-only training)* |
| Python | 3.12 |
| Conda env | `dr_env` |
| PyTorch | 2.4.1 (CPU) |
| Transformers | 4.50.0 |
| PEFT | 0.15.0 |
| TRL | 0.16.0 |
| llama-cpp-python | latest |

---

## Dataset

| Property | Value |
|---|---|
| Source | MMLU-Pro (Computer Science subset) |
| Train split | ~10 |
| Test split | ~137 |
| Split seed | fixed by `get_and_split_data.py` |
| Answer choices | A–J (10 options) |

---

## Run 1 — Baseline (No Training)

**Date:** 2026-04-25  
**Purpose:** Establish baseline accuracy of the unmodified student model.

### Model
| Parameter | Value |
|---|---|
| Model | Qwen2.5-1.5B-Instruct |
| Format | GGUF Q4_K_M |
| File | `models/Qwen2.5-1.5B-Instruct-GGUF/qwen2.5-1.5b-instruct-q4_k_m.gguf` |
| Inference | llama-cpp-python server, port 58000 |

### Inference Hyperparameters
| Parameter | Value |
|---|---|
| Temperature | 0.3 |
| Top-p | 0.9 |
| Max new tokens | 700 |
### Results
**Accuracy**  **17.52%** 


## Run 2 — D&R Distilled (10 Questions)

**Purpose:** Main experiment — distil D&R reasoning into the student model using 10 debate questions.

### Debate Configuration
| Parameter | Value |
|---|---|
| Debate questions | 10 |
| Debate rounds | 4 |
| Teacher 1 | GPT-4o (`gpt-4o-2024-08-06`) |
| Teacher 2 | Claude (`claude-opus-4-6`) |
| Teacher 3 | Gemini (`gemini-2.5-flash`) |
| Student | Qwen2.5-1.5B-Instruct Q4 (local llama.cpp) |
| Teacher temperature | 0.3 |
| Teacher max tokens | 1000 |
| Output | `data/MAG_new_mistral/MMLUProComp_1000.jsonl` |

### SFT Hyperparameters
| Parameter | Value |
|---|---|
| Base model | `Qwen/Qwen2.5-1.5B-Instruct` |
| Epochs | 2 |
| Learning rate | 2e-4 |
| Per-device batch size | 1 |
| Gradient accumulation steps | 4 |
| Effective batch size | 4 |
| Max sequence length | 512 |
| LoRA rank (r) | 16 |
| LoRA alpha | 32 |
| LoRA target | CAUSAL_LM |
| Optimizer | Adafactor |
| Dtype | float32 |
| Seed | 42 |
| Output | `tmp_sft/sft_chosen_MMLUProComp_mag_new_974_w_qwen_2epoch` |

### T-DPO Hyperparameters
| Parameter | Value |
|---|---|
| Epochs | 3 |
| Learning rate | 5e-6 |
| Per-device batch size | 1 |
| Gradient accumulation steps | 8 |
| Effective batch size | 8 |
| Max length | 512 |
| Max prompt length | 256 |
| Truncation mode | keep_end |
| Warmup ratio | 0.1 |
| Optimizer | Adafactor |
| Gradient checkpointing | Yes |
| Model adapter name | `tdpo` |
| Ref adapter name | `reference` |
| SFT adapter | `tmp_sft/sft_chosen_MMLUProComp_mag_new_974_w_qwen_2epoch` |
| Seed | 42 |
| Output | `tmp_tdpo/tdpo_after_sft_MMLUProComp_mag_new_974_w_qwen_3epoch` |

### Merge & Conversion
| Step | Details |
|---|---|
| Merge script | `replications/merge_adapter.py` |
| Merged output | `models/qwen-distilled-merged/` |
| GGUF output | `models/qwen-distilled.gguf` |
| GGUF type | f16 |
| GGUF size | 2.9 GB |

### Results
 **Accuracy** **22.63%** 

## Summary Table

| Run | Debate Qs | Model | Accuracy |
|---|---|---|---|---|---|
| Baseline | 0 | Qwen2.5-1.5B Q4 (base) |  17.52% |
| 10q distilled | 10 | Qwen2.5-1.5B (SFT+T-DPO) |  22.63% |

---

## Notes & Deviations from Original Paper

| Aspect | Original Paper | This Implementation | Reason |
|---|---|---|---|
| Student model | Mistral-7B-Instruct | Qwen2.5-1.5B-Instruct | 7B too large for CPU inference |
| Teacher 1 | Claude 3.5 Sonnet | Gpt-4o | Model availability |
| Teacher 2 | Claude 3.5 Sonnet | Claude Opus 4.6 | Model availability |
| Teacher 3 | Gemini 1.5 Flash | Gemini 2.5 Flash | Model availability |
| Inference backend | vLLM (GPU) | llama-cpp-python GGUF (CPU) | 6GB GPU insufficient for vLLM |
| Debate scale | Full training set (~137 Qs) | 10 | CPU time constraint |
| Max sequence length | 1024 / 512 | 512 / 256 | CPU RAM constraint |
| DDP accelerate | Yes | No (direct python) | No `ddp.yaml` config |
| GGUF quantization | N/A | f16 (full precision for distilled) | Q4 only for base |

---

## Reproducibility

All runs use **seed = 42** for both SFT and T-DPO training.

To replicate exactly, follow `replications/replication_guide.ipynb`.